# Padded Splines
The data samples are represented with circles. The first sample, as well as its replicates when the padding is globally periodic, is indicated by a red stem line. The portion of curve in thick green is tied to the observed data, with a number of samples that can be specified. The portion of curve in thick blue is the complement to a full period.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_degree = 5 # Maximal spline degree
max_samples = 12 # Maximal support of the observed data samples
# Random periodic cubic spline
k0 = 6 # Initial number of observed samples
s0 = sk.PeriodicSpline1D.from_spline_coeff(np.random.standard_normal(k0), degree = 3)

# Plot
def update_plot (
    degree = 3,
    samples = 6,
    padding = 0,
    display = 2
):
    global k0
    global s0

    # Update of the degree
    if s0.degree != degree:
        s0 = s0.projected(degree = degree)

    # Update of the number of samples
    f0 = s0.get_samples(0, support_length = k0)
    if k0 < samples:
        f0 = np.append(f0, np.random.standard_normal(samples - len(f0)))
    else:
        f0 = f0[ : samples]
    k0 = len(f0) # Number of observed samples

    highlight = sk.interval.Empty() # Support of the observed data
    downlight = sk.interval.Empty() # Complementary support
    plt_domain = sk.interval.Empty() # Support over three periods
    f = f0

    if 0 == padding: # Periodic
        plt_domain = sk.interval.ClosedOpen((-k0 - 0.5, 2 * k0 + 0.5))
        highlight = sk.interval.ClosedOpen((0, k0))
    elif 1 == padding: # Narrow Mirror
        f = np.zeros(max(1, 2 * k0 - 2), dtype = float)
        for k in range(len(f)):
            f[k] = sk.pad_n(f0, at = k)
        plt_domain = sk.interval.ClosedOpen((-len(f) - 0.5, 2 * len(f) + 0.5))
        highlight = sk.interval.ClosedOpen((0, max(1, k0 - 1)))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, len(f)))
    elif 2 == padding: # Wide Mirror
        f = np.zeros(2 * k0, dtype = float)
        for k in range(len(f)):
            f[k] = sk.pad_w(f0, at = k)
        plt_domain = sk.interval.ClosedOpen((-len(f) - 0.5, 2 * len(f) + 0.5))
        highlight = sk.interval.ClosedOpen((-0.5, k0 - 0.5))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, len(f) - 0.5))
    elif 3 == padding: # Anti-Mirror
        f = np.zeros(max_degree + 1 + 3 * (2 * k0 - 2) + max_degree + 1, dtype = float)
        delay = max_degree + 1 + 2 * k0 - 2
        for k in range(len(f)):
            f[k] = sk.pad_a(f0, at = k - delay)
        f = np.roll(f, -delay)
        plt_domain = sk.interval.ClosedOpen((-2 * k0 + 2 - 0.5, 4 * k0 - 4 + 0.5))
        highlight = sk.interval.ClosedOpen((0, max(1, k0 - 1)))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, 2 * k0 - 2))
    elif 4 == padding: # Nega-Periodic
        f = np.zeros(2 * k0, dtype = float)
        for k in range(len(f)):
            f[k] = sk.pad_np(f0, at = k)
        plt_domain = sk.interval.ClosedOpen((-len(f) - 0.5, 2 * len(f) + 0.5))
        highlight = sk.interval.ClosedOpen((0, k0))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, len(f)))
    elif 5 == padding: # Nega-Narrow Mirror
        f = np.zeros(2 * k0 + 2, dtype = float)
        for k in range(len(f)):
            f[k] = sk.pad_nn(f0, at = k)
        plt_domain = sk.interval.ClosedOpen((-len(f) - 0.5, 2 * len(f) + 0.5))
        highlight = sk.interval.ClosedOpen((-1, k0))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, len(f) - 1))
    elif 6 == padding: # Nega-Wide Mirror
        f = np.zeros(2 * k0, dtype = float)
        for k in range(len(f)):
            f[k] = sk.pad_nw(f0, at = k)
        plt_domain = sk.interval.ClosedOpen((-len(f) - 0.5, 2 * len(f) + 0.5))
        highlight = sk.interval.ClosedOpen((-0.5, k0 - 0.5))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, len(f) - 0.5))
    s0 = sk.PeriodicSpline1D.from_samples(f, degree = degree)

    (fig, ax) = plt.subplots()
    if 0 == display: # Show the observed samples
        s0.plot(
            (fig, ax),
            plotdomain = sk.interval.ClosedOpen((-0.5, k0 - 0.5)),
            plotpoints = 101,
            curve_fmt = "-C2",
            curve_lw = 3,
            knot_marker = "None"
        )
    elif 1 == display: # Show one period
        # Determine the range of the plot
        s0.plot(
            (fig, ax),
            plotdomain = list(highlight | downlight)[0],
            plotpoints = 301,
            knot_marker = "None"
        )
        s0.plot(
            (fig, ax),
            plotdomain = highlight,
            plotpoints = 101,
            curve_fmt = "-C2",
            curve_lw = 3,
            knot_marker = "None"
        )
        if 0 < downlight.diameter:
            s0.plot(
                (fig, ax),
                plotdomain = downlight,
                plotpoints = 101,
                curve_fmt = "-C0",
                curve_lw = 3,
                knot_marker = "None"
            )
    elif 2 == display: # Show three periods
        s0.plot((fig, ax), plotdomain = plt_domain, plotpoints = 301, knot_marker = "None")
        s0.plot(
            (fig, ax),
            plotdomain = highlight,
            plotpoints = 101,
            curve_fmt = "-C2",
            curve_lw = 3,
            knot_marker = "None"
        )
        if 0 < downlight.diameter:
            s0.plot(
                (fig, ax),
                plotdomain = downlight,
                plotpoints = 101,
                curve_fmt = "-C0",
                curve_lw = 3,
                knot_marker = "None"
            )
    plt.show()

widgets.interactive(
    update_plot,
    degree = (0, max_degree),
    samples = (1, max_samples),
    padding = widgets.RadioButtons(
        options = [
            ("Periodic", 0),
            ("Narrow Mirror", 1),
            ("Wide Mirror", 2),
            ("Anti-Mirror", 3),
            ("Nega-Periodic", 4),
            ("Nega-Narrow Mirror", 5),
            ("Nega-Wide Mirror", 6)
        ],
        value = 0,
        description = "Padding:",
        disabled = False
    ),
    display = widgets.RadioButtons(
        options = [
            ("Observed Samples", 0),
            ("One Period", 1),
            ("Three Periods", 2)
        ],
        value = 2,
        description = "Displayed Support:",
        disabled = False
    )
)
